<a href="https://colab.research.google.com/github/davechang-99/Dementia_EN/blob/main/ADReSS_Dementia_Detection_friedman_vit_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ADReSS 2020: Alzheimer's Dementia Recognition through Spontaneous Speech

## Overview
This notebook implements a comprehensive analysis pipeline for the ADReSS Challenge 2020 dataset to detect Alzheimer's dementia through spontaneous speech analysis.

### Dataset Information
- **Training Data**: 108 participants (54 control + 54 dementia)
- **Test Data**: 48 participants (unlabeled)
- **Tasks**:
  1. Binary Classification (Control vs Dementia)
  2. MMSE Score Regression

### Features Extracted
1. **Audio Features**: MFCC, Spectral features, Prosodic features
2. **Linguistic Features**: Vocabulary diversity, Syntactic complexity, Semantic features
3. **Temporal Features**: Pause patterns, Speech rate, Disfluencies


## 1. Environment Setup and Library Installation


In [ ]:
# Install required packages
%pip install librosa soundfile scikit-learn pandas numpy matplotlib seaborn
%pip install textstat nltk spacy transformers torch
%pip install plotly wordcloud
%pip install -U transformers torch torchvision pillow

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Install spaCy model
!python -m spacy download en_core_web_sm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 146.3 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...


KeyboardInterrupt: 

In [ ]:
# Import libraries
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_squared_error, r2_score
from sklearn.decomposition import PCA
import librosa
import soundfile as sf
import re
import textstat
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter
import spacy
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Environment setup completed!")


## 2. Data Loading and Preprocessing


In [ ]:
# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

# Set data paths - Update these paths according to your Google Drive structure
BASE_PATH = '/content/drive/MyDrive/project'
TRAIN_PATH = f'{BASE_PATH}/ADReSS-IS2020-train/ADReSS-IS2020-data/train'
TEST_PATH = f'{BASE_PATH}/ADReSS-IS2020-test/ADReSS-IS2020-data/test'

print(f"Base path: {BASE_PATH}")
print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")


In [ ]:
def load_metadata(file_path):
    """Load metadata from text file"""
    import pandas as pd  # Import pandas here to avoid NameError

    data = []
    with open(file_path, 'r') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            if not line:  # Skip empty lines
                continue

            parts = line.split(';')
            if len(parts) >= 3:
                subject_id = parts[0].strip()

                # Skip header line (check if first column contains 'ID' or if age is not numeric)
                if subject_id.upper() in ['ID', 'SUBJECT_ID'] or line_num == 0:
                    try:
                        int(parts[1].strip())  # Test if age is numeric
                    except ValueError:
                        continue  # Skip header line

                try:
                    age = int(parts[1].strip())
                    gender = parts[2].strip()

                    # Handle MMSE field - check for NA, empty, or missing values
                    mmse = None
                    if len(parts) > 3:
                        mmse_str = parts[3].strip()
                        if mmse_str and mmse_str.upper() not in ['NA', 'NULL', '']:
                            try:
                                mmse = int(mmse_str)
                            except ValueError:
                                mmse = None

                    data.append({
                        'subject_id': subject_id,
                        'age': age,
                        'gender': gender,
                        'mmse': mmse
                    })
                except ValueError as e:
                    print(f"Skipping line due to parsing error: {line}")
                    continue

    return pd.DataFrame(data)

# Load training metadata
cc_meta = load_metadata(f'{TRAIN_PATH}/cc_meta_data.txt')
cd_meta = load_metadata(f'{TRAIN_PATH}/cd_meta_data.txt')

# Add labels
cc_meta['label'] = 0  # Control
cd_meta['label'] = 1  # Dementia

# Combine training metadata
train_meta = pd.concat([cc_meta, cd_meta], ignore_index=True)

# Load test metadata
test_meta = load_metadata(f'{TEST_PATH}/meta_data.txt')

print(f"Training data: {len(train_meta)} subjects")
print(f"Test data: {len(test_meta)} subjects")
print("\nTraining data distribution:")
print(train_meta['label'].value_counts())


## 3. Data Exploration and Visualization


In [ ]:
# Data exploration
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution by group
sns.boxplot(data=train_meta, x='label', y='age', ax=axes[0,0])
axes[0,0].set_title('Age Distribution by Group')
axes[0,0].set_xticklabels(['Control', 'Dementia'])

# Gender distribution by group
pd.crosstab(train_meta['label'], train_meta['gender']).plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Gender Distribution by Group')
axes[0,1].set_xticklabels(['Control', 'Dementia'], rotation=0)

# MMSE score distribution
train_meta_mmse = train_meta.dropna(subset=['mmse'])
sns.boxplot(data=train_meta_mmse, x='label', y='mmse', ax=axes[1,0])
axes[1,0].set_title('MMSE Score Distribution by Group')
axes[1,0].set_xticklabels(['Control', 'Dementia'])

# Age vs MMSE scatter plot
colors = ['blue' if x == 0 else 'red' for x in train_meta_mmse['label']]
axes[1,1].scatter(train_meta_mmse['age'], train_meta_mmse['mmse'], c=colors, alpha=0.6)
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('MMSE Score')
axes[1,1].set_title('Age vs MMSE Score')
axes[1,1].legend(['Control', 'Dementia'])

plt.tight_layout()
plt.show()

# Statistical summary
print("\nStatistical Summary:")
print(train_meta.groupby('label')[['age', 'mmse']].describe())


## 4. Audio Feature Extraction


## 4.5. Vision Transformer for Audio Analysis


In [ ]:
# Compatible with latest transformers versions
# Install additional packages for ViT
%pip install transformers torch torchvision pillow

import torch
import torch.nn as nn
from torchvision import transforms
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import matplotlib.pyplot as plt

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def create_spectrogram_image(audio_path, target_size=(224, 224)):
    """
    Create spectrogram image from audio file for ViT input
    """
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=22050, duration=30)  # Limit to 30 seconds

        # Create mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=128, fmax=8000, hop_length=512
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize to 0-255 range
        mel_spec_norm = ((mel_spec_db - mel_spec_db.min()) /
                        (mel_spec_db.max() - mel_spec_db.min()) * 255).astype(np.uint8)

        # Convert to 3-channel image (RGB)
        spectrogram_rgb = np.stack([mel_spec_norm] * 3, axis=-1)

        # Resize to target size
        pil_image = Image.fromarray(spectrogram_rgb)
        pil_image = pil_image.resize(target_size, Image.Resampling.LANCZOS)

        return pil_image

    except Exception as e:
        print(f"Error creating spectrogram for {audio_path}: {e}")
        # Return a blank image in case of error
        return Image.new('RGB', target_size, color=(128, 128, 128))

def extract_vit_features(audio_path, model, feature_extractor):
    """
    Extract features from audio using ViT model
    """
    try:
        # Create spectrogram image
        spec_image = create_spectrogram_image(audio_path)

        # Prepare inputs for ViT
        inputs = feature_extractor(images=spec_image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Extract features (before classification head)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Use the last hidden state, average pooled
            features = outputs.hidden_states[-1].mean(dim=1).squeeze()

        return features.cpu().numpy()

    except Exception as e:
        print(f"Error extracting ViT features for {audio_path}: {e}")
        return np.zeros(768)  # Return zero vector if error

print("ViT feature extraction functions defined.")


## 4.6. Improved ViT Feature Extraction


In [ ]:
# Improved ViT feature extraction with better preprocessing and dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

def create_enhanced_spectrogram(audio_path, target_size=(224, 224)):
    """
    Create enhanced spectrogram with better preprocessing for ViT
    """
    try:
        # Load audio with better parameters
        y, sr = librosa.load(audio_path, sr=16000, duration=20)  # 16kHz, 20 seconds

        # Apply noise reduction (simple high-pass filter)
        y = librosa.effects.preemphasis(y)

        # Create multiple spectrograms for ensemble
        # 1. Mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=224, fmax=8000, hop_length=256, n_fft=1024
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # 2. Chroma features
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=256)
        chroma_resized = np.resize(chroma, (224, mel_spec_db.shape[1]))

        # 3. Spectral contrast
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=256)
        contrast_resized = np.resize(contrast, (224, mel_spec_db.shape[1]))

        # Resize temporal dimension to 224
        target_time = 224
        if mel_spec_db.shape[1] != target_time:
            mel_spec_db = np.resize(mel_spec_db, (224, target_time))
            chroma_resized = np.resize(chroma_resized, (224, target_time))
            contrast_resized = np.resize(contrast_resized, (224, target_time))

        # Normalize each channel separately
        def normalize_channel(data):
            return ((data - data.min()) / (data.max() - data.min() + 1e-8) * 255).astype(np.uint8)

        mel_norm = normalize_channel(mel_spec_db)
        chroma_norm = normalize_channel(chroma_resized)
        contrast_norm = normalize_channel(contrast_resized)

        # Combine as RGB channels
        spectrogram_rgb = np.stack([mel_norm, chroma_norm, contrast_norm], axis=-1)

        # Convert to PIL Image
        pil_image = Image.fromarray(spectrogram_rgb)

        return pil_image

    except Exception as e:
        print(f"Error creating enhanced spectrogram for {audio_path}: {e}")
        return Image.new('RGB', target_size, color=(128, 128, 128))

def extract_improved_vit_features(audio_path, model, feature_extractor, use_pca=True, n_components=50):
    """
    Extract improved ViT features with PCA dimensionality reduction
    """
    try:
        # Create enhanced spectrogram
        spec_image = create_enhanced_spectrogram(audio_path)

        # Data augmentation - create multiple views
        transforms_list = [
            spec_image,  # Original
            spec_image.transpose(Image.Transpose.FLIP_LEFT_RIGHT),  # Horizontal flip
        ]

        all_features = []

        for img in transforms_list:
            # Prepare inputs for ViT
            inputs = feature_extractor(images=img, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Extract features
            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)
                # Use multiple layers for richer features
                last_hidden = outputs.hidden_states[-1].mean(dim=1).squeeze()  # Last layer
                second_last = outputs.hidden_states[-2].mean(dim=1).squeeze()  # Second last layer

                # Combine features from multiple layers
                combined_features = torch.cat([last_hidden, second_last], dim=0)
                all_features.append(combined_features.cpu().numpy())

        # Average features from different augmentations
        final_features = np.mean(all_features, axis=0)

        # Apply PCA for dimensionality reduction if requested
        if use_pca and hasattr(extract_improved_vit_features, 'pca_model'):
            final_features = extract_improved_vit_features.pca_model.transform([final_features])[0]

        return final_features

    except Exception as e:
        print(f"Error extracting improved ViT features for {audio_path}: {e}")
        return np.zeros(n_components if use_pca else 1536)  # 768*2 for two layers

def setup_vit_pca(train_audio_paths, vit_model, feature_extractor, n_components=50):
    """
    Setup PCA for ViT features using training data
    """
    print("Setting up PCA for ViT features...")

    # Extract features from a subset of training data
    sample_features = []
    max_samples = min(50, len(train_audio_paths))  # Use max 50 samples for PCA fitting

    for i, audio_path in enumerate(train_audio_paths[:max_samples]):
        if i % 10 == 0:
            print(f"Processing {i+1}/{max_samples} for PCA...")

        # Extract raw features (without PCA)
        spec_image = create_enhanced_spectrogram(audio_path)
        inputs = feature_extractor(images=spec_image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = vit_model(**inputs, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1].mean(dim=1).squeeze()
            second_last = outputs.hidden_states[-2].mean(dim=1).squeeze()
            combined_features = torch.cat([last_hidden, second_last], dim=0)
            sample_features.append(combined_features.cpu().numpy())

    # Fit PCA
    pca = PCA(n_components=n_components, random_state=42)
    pca.fit(sample_features)

    # Store PCA model as function attribute
    extract_improved_vit_features.pca_model = pca

    explained_variance = np.sum(pca.explained_variance_ratio_)
    print(f"PCA setup complete. {n_components} components explain {explained_variance:.3f} of variance.")

    return pca

print("Improved ViT feature extraction functions defined.")


In [ ]:
# Compatible with latest transformers versions
# Setup improved ViT feature extraction
# First check if ViT model variables exist, if not, initialize them
if 'vit_model' not in globals() or 'feature_extractor' not in globals():
    print("ViT model not loaded yet. Loading now...")
    try:
        from transformers import ViTImageProcessor, ViTForImageClassification
        feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
        vit_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
        vit_model = vit_model.to(device)
        vit_model.eval()
        print("ViT model loaded successfully!")
    except Exception as e:
        print(f"Error loading ViT model: {e}")
        vit_model = None
        feature_extractor = None

if vit_model is not None and feature_extractor is not None:
    print("Setting up improved ViT feature extraction...")

    # Get training audio paths for PCA setup
    train_audio_paths = []

    # Check if required variables exist
    if 'train_meta' in globals() and 'TRAIN_PATH' in globals():
        for idx, row in train_meta.iterrows():
            subject_id = row['subject_id']
            group_folder = 'cc' if row['label'] == 0 else 'cd'
            audio_path = f"{TRAIN_PATH}/Full_wave_enhanced_audio/{group_folder}/{subject_id}.wav"
            if os.path.exists(audio_path):
                train_audio_paths.append(audio_path)
    else:
        print("Required variables (train_meta, TRAIN_PATH) not available.")
        print("Please run the following sections first:")
        print("1. Data Loading and Preprocessing")
        print("2. Environment Setup")
        train_audio_paths = []

    if len(train_audio_paths) > 0:
        # Setup PCA for ViT features
        vit_pca = setup_vit_pca(train_audio_paths, vit_model, feature_extractor, n_components=50)
        print("Improved ViT setup completed!")
    else:
        print("No training audio files found for ViT setup.")
        vit_pca = None
else:
    print("ViT model not available, skipping improved setup.")
    vit_pca = None

def extract_features_for_dataset_improved(data_path, metadata, group_type='train', use_improved_vit=True):
    """
    Improved feature extraction with better ViT features
    """
    features_list = []

    for idx, row in metadata.iterrows():
        subject_id = row['subject_id']
        print(f"Processing {subject_id} (improved)...")

        feature_dict = {
            'subject_id': subject_id,
            'age': row['age'],
            'gender': 1 if row['gender'] == 'male' else 0,
        }

        if group_type == 'train':
            feature_dict['label'] = row['label']
            if pd.notna(row['mmse']):
                feature_dict['mmse'] = row['mmse']

        # Audio features
        if group_type == 'train':
            group_folder = 'cc' if row['label'] == 0 else 'cd'
            audio_path = f"{data_path}/Full_wave_enhanced_audio/{group_folder}/{subject_id}.wav"
            transcription_path = f"{data_path}/transcription/{group_folder}/{subject_id}.cha"
        else:
            audio_path = f"{data_path}/Full_wave_enhanced_audio/{subject_id}.wav"
            transcription_path = f"{data_path}/transcription/{subject_id}.cha"

        if os.path.exists(audio_path):
            # Traditional audio features
            audio_features = extract_audio_features(audio_path)
            if audio_features:
                feature_dict.update(audio_features)

            # Improved ViT features
            if (use_improved_vit and vit_model is not None and
                feature_extractor is not None and vit_pca is not None):
                try:
                    vit_features = extract_improved_vit_features(
                        audio_path, vit_model, feature_extractor, use_pca=True, n_components=50
                    )
                    # Add improved ViT features with prefix
                    for i, feat in enumerate(vit_features):
                        feature_dict[f'vit_improved_{i}'] = feat
                except Exception as e:
                    print(f"Error with improved ViT for {subject_id}: {e}")

        # Linguistic features
        if os.path.exists(transcription_path):
            text = read_cha_file(transcription_path)
            linguistic_features = extract_linguistic_features(text)
            feature_dict.update(linguistic_features)

        features_list.append(feature_dict)

    return pd.DataFrame(features_list)

print("Improved feature extraction pipeline ready.")


In [ ]:
# Compatible with latest transformers versions
# Initialize ViT model for feature extraction
print("Loading ViT model...")
try:
    feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
    vit_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    vit_model = vit_model.to(device)
    vit_model.eval()
    print("ViT model loaded successfully!")
except Exception as e:
    print(f"Error loading ViT model: {e}")
    print("Will skip ViT features in analysis")
    vit_model = None
    feature_extractor = None

def visualize_spectrogram_samples(audio_paths, labels, num_samples=4):
    """
    Visualize spectrogram samples for different classes
    """
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))

    control_paths = [path for path, label in zip(audio_paths, labels) if label == 0][:num_samples]
    dementia_paths = [path for path, label in zip(audio_paths, labels) if label == 1][:num_samples]

    # Plot control samples
    for i, audio_path in enumerate(control_paths):
        spec_image = create_spectrogram_image(audio_path)
        axes[0, i].imshow(spec_image)
        axes[0, i].set_title(f'Control {i+1}')
        axes[0, i].axis('off')

    # Plot dementia samples
    for i, audio_path in enumerate(dementia_paths):
        spec_image = create_spectrogram_image(audio_path)
        axes[1, i].imshow(spec_image)
        axes[1, i].set_title(f'Dementia {i+1}')
        axes[1, i].axis('off')

    plt.suptitle('Mel Spectrogram Comparison: Control vs Dementia')
    plt.tight_layout()
    plt.show()

print("ViT visualization functions defined.")


In [ ]:
def extract_audio_features(audio_path):
    """Extract comprehensive audio features from audio file"""
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=None)

        # Basic audio properties
        duration = librosa.get_duration(y=y, sr=sr)

        # MFCC features (13 coefficients)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfccs, axis=1)
        mfcc_std = np.std(mfccs, axis=1)

        # Spectral features
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        zero_crossing_rate = librosa.feature.zero_crossing_rate(y)[0]

        # Prosodic features
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
        pitch_values = []
        for t in range(pitches.shape[1]):
            index = magnitudes[:, t].argmax()
            pitch = pitches[index, t]
            if pitch > 0:
                pitch_values.append(pitch)

        # Energy features
        rms_energy = librosa.feature.rms(y=y)[0]

        # Combine all features
        features = {
            'duration': duration,
            'spectral_centroid_mean': np.mean(spectral_centroids),
            'spectral_centroid_std': np.std(spectral_centroids),
            'spectral_rolloff_mean': np.mean(spectral_rolloff),
            'spectral_rolloff_std': np.std(spectral_rolloff),
            'spectral_bandwidth_mean': np.mean(spectral_bandwidth),
            'spectral_bandwidth_std': np.std(spectral_bandwidth),
            'zero_crossing_rate_mean': np.mean(zero_crossing_rate),
            'zero_crossing_rate_std': np.std(zero_crossing_rate),
            'rms_energy_mean': np.mean(rms_energy),
            'rms_energy_std': np.std(rms_energy),
            'pitch_mean': np.mean(pitch_values) if pitch_values else 0,
            'pitch_std': np.std(pitch_values) if pitch_values else 0,
        }

        # Add MFCC features
        for i in range(13):
            features[f'mfcc_{i}_mean'] = mfcc_mean[i]
            features[f'mfcc_{i}_std'] = mfcc_std[i]

        return features

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

print("Audio feature extraction function defined.")


## 5. Text Analysis and Linguistic Features


In [ ]:
# Load spaCy model for linguistic analysis
nlp = spacy.load('en_core_web_sm')

# Ensure NLTK data is available
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

# Test punkt tokenizer
try:
    sent_tokenize("Test sentence.")
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    from nltk.tokenize import sent_tokenize

def read_cha_file(file_path):
    """Read CHAT format transcription file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Extract participant speech (lines starting with *PAR:)
        participant_lines = []
        for line in content.split('\n'):
            if line.startswith('*PAR:'):
                text = line[5:].strip()  # Remove *PAR: prefix
                # Clean up CHAT annotations
                text = re.sub(r'\[.*?\]', '', text)  # Remove annotations
                text = re.sub(r'\(.*?\)', '', text)  # Remove parenthetical comments
                text = re.sub(r'[+<>]', '', text)   # Remove CHAT symbols
                text = re.sub(r'\s+', ' ', text)    # Normalize whitespace
                if text.strip():
                    participant_lines.append(text.strip())

        return ' '.join(participant_lines)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

def extract_linguistic_features(text):
    """Extract comprehensive linguistic features from text"""
    if not text.strip():
        return {}

    # Basic text statistics
    doc = nlp(text)
    sentences = sent_tokenize(text)
    words = word_tokenize(text.lower())
    words_clean = [w for w in words if w.isalpha() and w not in stop_words]

    # Vocabulary and lexical diversity
    vocabulary_size = len(set(words_clean))
    total_words = len(words_clean)
    type_token_ratio = vocabulary_size / total_words if total_words > 0 else 0

    # Readability scores
    flesch_reading_ease = textstat.flesch_reading_ease(text)
    flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)

    # Syntactic complexity
    avg_sentence_length = np.mean([len(word_tokenize(s)) for s in sentences]) if sentences else 0

    # POS tag analysis
    pos_counts = Counter([token.pos_ for token in doc])
    noun_ratio = pos_counts['NOUN'] / len(doc) if len(doc) > 0 else 0
    verb_ratio = pos_counts['VERB'] / len(doc) if len(doc) > 0 else 0
    adj_ratio = pos_counts['ADJ'] / len(doc) if len(doc) > 0 else 0

    # Semantic features
    named_entities = len(doc.ents)

    # Disfluency markers (approximation)
    disfluencies = len(re.findall(r'\b(um|uh|er|ah)\b', text.lower()))
    repetitions = len(re.findall(r'\b(\w+)\s+\1\b', text.lower()))

    features = {
        'text_length': len(text),
        'word_count': total_words,
        'sentence_count': len(sentences),
        'vocabulary_size': vocabulary_size,
        'type_token_ratio': type_token_ratio,
        'avg_sentence_length': avg_sentence_length,
        'flesch_reading_ease': flesch_reading_ease,
        'flesch_kincaid_grade': flesch_kincaid_grade,
        'noun_ratio': noun_ratio,
        'verb_ratio': verb_ratio,
        'adj_ratio': adj_ratio,
        'named_entities': named_entities,
        'disfluencies': disfluencies,
        'repetitions': repetitions
    }

    return features

print("Linguistic feature extraction functions defined.")


## 6. Feature Extraction Pipeline


In [ ]:
def extract_features_for_dataset(data_path, metadata, group_type='train'):
    """Extract all features for a dataset"""
    features_list = []

    for idx, row in metadata.iterrows():
        subject_id = row['subject_id']
        print(f"Processing {subject_id}...")

        feature_dict = {
            'subject_id': subject_id,
            'age': row['age'],
            'gender': 1 if row['gender'] == 'male' else 0,
        }

        if group_type == 'train':
            feature_dict['label'] = row['label']
            if pd.notna(row['mmse']):
                feature_dict['mmse'] = row['mmse']

        # Audio features
        if group_type == 'train':
            group_folder = 'cc' if row['label'] == 0 else 'cd'
            audio_path = f"{data_path}/Full_wave_enhanced_audio/{group_folder}/{subject_id}.wav"
            transcription_path = f"{data_path}/transcription/{group_folder}/{subject_id}.cha"
        else:
            audio_path = f"{data_path}/Full_wave_enhanced_audio/{subject_id}.wav"
            transcription_path = f"{data_path}/transcription/{subject_id}.cha"

        if os.path.exists(audio_path):
            # Traditional audio features
            audio_features = extract_audio_features(audio_path)
            if audio_features:
                feature_dict.update(audio_features)

            # ViT features if model is available
            if vit_model is not None and feature_extractor is not None:
                vit_features = extract_vit_features(audio_path, vit_model, feature_extractor)
                # Add ViT features with prefix
                for i, feat in enumerate(vit_features):
                    feature_dict[f'vit_{i}'] = feat

        # Linguistic features
        if os.path.exists(transcription_path):
            text = read_cha_file(transcription_path)
            linguistic_features = extract_linguistic_features(text)
            feature_dict.update(linguistic_features)

        features_list.append(feature_dict)

    return pd.DataFrame(features_list)

# Extract features for training data (this may take several minutes)
print("Extracting features for training data...")
train_features = extract_features_for_dataset(TRAIN_PATH, train_meta, 'train')
print(f"\nTraining features extracted: {train_features.shape}")
print("Feature columns:", list(train_features.columns))


## 7. Machine Learning Models


In [ ]:
# Visualize spectrograms for sample data
if len(train_features) > 0:
    # Get audio paths for visualization
    sample_audio_paths = []
    sample_labels = []

    for idx, row in train_meta.head(20).iterrows():  # Sample first 20
        subject_id = row['subject_id']
        label = row['label']
        group_folder = 'cc' if label == 0 else 'cd'
        audio_path = f"{TRAIN_PATH}/Full_wave_enhanced_audio/{group_folder}/{subject_id}.wav"

        if os.path.exists(audio_path):
            sample_audio_paths.append(audio_path)
            sample_labels.append(label)

    if len(sample_audio_paths) >= 8:  # Need at least 8 samples (4 per class)
        print("Visualizing spectrogram samples...")
        visualize_spectrogram_samples(sample_audio_paths, sample_labels)

    print(f"Total features including ViT: {train_features.shape[1]}")

    # Analyze feature types
    feature_types = {
        'demographic': 0,
        'traditional_audio': 0,
        'linguistic': 0,
        'vit': 0
    }

    for col in train_features.columns:
        if col in ['age', 'gender']:
            feature_types['demographic'] += 1
        elif col.startswith('vit_'):
            feature_types['vit'] += 1
        elif col in ['text_length', 'word_count', 'vocabulary_size', 'type_token_ratio',
                     'flesch_reading_ease', 'noun_ratio', 'verb_ratio']:
            feature_types['linguistic'] += 1
        elif col not in ['subject_id', 'label', 'mmse']:
            feature_types['traditional_audio'] += 1

    print("\nFeature breakdown:")
    for feat_type, count in feature_types.items():
        print(f"- {feat_type.replace('_', ' ').title()}: {count} features")
else:
    print("No training features available for visualization")


In [ ]:
# Prepare data for modeling
feature_columns = [col for col in train_features.columns
                  if col not in ['subject_id', 'label', 'mmse']]

X = train_features[feature_columns]
y_class = train_features['label']
y_mmse = train_features['mmse'].dropna()
X_mmse = X.loc[y_mmse.index]

# Handle missing values
X_filled = X.fillna(X.mean())
X_mmse_filled = X_mmse.fillna(X_mmse.mean())

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
X_mmse_scaled = scaler.transform(X_mmse_filled)

# Split data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class
)

X_train_mmse, X_val_mmse, y_train_mmse, y_val_mmse = train_test_split(
    X_mmse_scaled, y_mmse, test_size=0.2, random_state=42
)

print(f"Classification data shape: {X_filled.shape}")
print(f"Regression data shape: {X_mmse_filled.shape}")
print(f"Number of features: {len(feature_columns)}")


### 7.1 Classification Models (Control vs Dementia)


In [ ]:
# Define classification models
classification_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

# Train and evaluate classification models
classification_results = {}

for name, model in classification_models.items():
    print(f"\nTraining {name}...")

    # Cross-validation
    cv_scores = cross_val_score(model, X_scaled, y_class, cv=5, scoring='accuracy')

    # Train on training set
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_val)

    # Evaluation metrics
    accuracy = accuracy_score(y_val, y_pred)

    classification_results[name] = {
        'model': model,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'val_accuracy': accuracy,
        'predictions': y_pred
    }

    print(f"CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")
    print(f"Validation Accuracy: {accuracy:.3f}")
    print("Classification Report:")
    print(classification_report(y_val, y_pred, target_names=['Control', 'Dementia']))


### 7.1.1 Friedman Test for Statistical Significance Across Classification Models

This section performs a **Friedman test** on the fold-wise classification accuracies of **Logistic Regression, SVM, and Random Forest**.  
The test is appropriate here because the same 5 cross-validation folds are used to compare multiple models.

**What this section does**
1. Recomputes fold-wise accuracy scores using a shared `StratifiedKFold`
2. Creates a fold-by-model accuracy table
3. Runs the Friedman test
4. Saves the results to CSV for reporting in the paper

> Note: This section uses a `Pipeline(StandardScaler + model)` inside cross-validation so that scaling is fit only on the training fold, which is safer for statistical comparison.


In [ ]:
# Friedman test for model comparison (Classification)
from scipy.stats import friedmanchisquare
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone

print("Running Friedman test on 5-fold classification accuracies...")

# Use the same models as the classification section
friedman_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Shared folds across all models
friedman_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use the unscaled filled feature matrix so scaling happens safely inside CV
X_friedman = X_filled.copy()
y_friedman = y_class.copy()

fold_score_dict = {}

for model_name, model in friedman_models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', clone(model))
    ])

    cv_result = cross_validate(
        pipeline,
        X_friedman,
        y_friedman,
        cv=friedman_cv,
        scoring='accuracy',
        return_train_score=False,
        n_jobs=-1
    )

    fold_score_dict[model_name] = cv_result['test_score']
    print(f"\n{model_name} fold accuracies:")
    for i, score in enumerate(cv_result['test_score'], start=1):
        print(f"  Fold {i}: {score:.4f}")
    print(f"  Mean ± SD: {cv_result['test_score'].mean():.4f} ± {cv_result['test_score'].std(ddof=1):.4f}")

# Create a fold-wise accuracy table
friedman_scores_df = pd.DataFrame(fold_score_dict, index=[f'Fold {i}' for i in range(1, 6)])
friedman_scores_df.index.name = 'Fold'

print("\nFold-wise accuracy table:")
display(friedman_scores_df)

# Run the Friedman test
friedman_stat, friedman_p = friedmanchisquare(
    friedman_scores_df['Logistic Regression'].values,
    friedman_scores_df['SVM'].values,
    friedman_scores_df['Random Forest'].values
)

friedman_summary_df = pd.DataFrame({
    'Test': ['Friedman Test'],
    'Statistic': [friedman_stat],
    'p_value': [friedman_p],
    'Significant_at_0.05': ['Yes' if friedman_p < 0.05 else 'No']
})

print("\nFriedman test result:")
display(friedman_summary_df)

if friedman_p < 0.05:
    print("Interpretation: The classification models show a statistically significant difference across the 5 folds (p < 0.05).")
else:
    print("Interpretation: No statistically significant difference was detected across the 5 folds (p >= 0.05).")

# Save outputs for paper/report writing
friedman_scores_df.to_csv('friedman_fold_accuracies_classification.csv')
friedman_summary_df.to_csv('friedman_test_summary_classification.csv', index=False)

print("\nSaved files:")
print("- friedman_fold_accuracies_classification.csv")
print("- friedman_test_summary_classification.csv")


### 7.4 Improved ViT Performance Testing


In [ ]:
# Test improved ViT features
print("Extracting improved features for comparison...")

# Extract features with improved ViT
train_features_improved = extract_features_for_dataset_improved(TRAIN_PATH, train_meta, 'train')

# Prepare data for comparison
feature_columns_improved = [col for col in train_features_improved.columns
                           if col not in ['subject_id', 'label', 'mmse']]

X_improved = train_features_improved[feature_columns_improved]
y_class_improved = train_features_improved['label']

# Handle missing values
X_improved_filled = X_improved.fillna(X_improved.mean())

def compare_vit_improvements(X_original, X_improved, y_class):
    """
    Compare original vs improved ViT features
    """
    results = {}
    scaler = StandardScaler()

    # Test different ViT feature combinations
    feature_sets = {}

    # 1. Original ViT features only
    original_vit_cols = [col for col in X_original.columns if col.startswith('vit_') and not col.startswith('vit_improved_')]
    if len(original_vit_cols) > 0:
        feature_sets['Original ViT'] = X_original[original_vit_cols]

    # 2. Improved ViT features only
    improved_vit_cols = [col for col in X_improved.columns if col.startswith('vit_improved_')]
    if len(improved_vit_cols) > 0:
        feature_sets['Improved ViT'] = X_improved[improved_vit_cols]

    # 3. Traditional + Linguistic features (baseline)
    traditional_cols = [col for col in X_improved.columns
                       if not col.startswith('vit_')
                       and col not in ['text_length', 'word_count', 'vocabulary_size', 'type_token_ratio',
                                     'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'sentence_count',
                                     'avg_sentence_length', 'flesch_kincaid_grade', 'adj_ratio',
                                     'named_entities', 'disfluencies', 'repetitions']]

    linguistic_cols = [col for col in X_improved.columns
                      if col in ['text_length', 'word_count', 'vocabulary_size', 'type_token_ratio',
                                'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'sentence_count',
                                'avg_sentence_length', 'flesch_kincaid_grade', 'adj_ratio',
                                'named_entities', 'disfluencies', 'repetitions']]

    baseline_cols = traditional_cols + linguistic_cols
    feature_sets['Traditional + Linguistic'] = X_improved[baseline_cols]

    # 4. Traditional + Linguistic + Improved ViT
    if len(improved_vit_cols) > 0:
        all_improved_cols = baseline_cols + improved_vit_cols
        feature_sets['All + Improved ViT'] = X_improved[all_improved_cols]

    # Test each feature set
    for name, features in feature_sets.items():
        if features.shape[1] == 0:
            continue

        print(f"\nTesting {name} ({features.shape[1]} features)...")

        # Scale features
        X_scaled = scaler.fit_transform(features.fillna(features.mean()))

        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        cv_scores = cross_val_score(rf, X_scaled, y_class, cv=5, scoring='accuracy')

        results[name] = {
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'n_features': features.shape[1]
        }

        print(f"CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    return results

# Run improved comparison
if 'train_features_improved' in locals() and len(train_features_improved) > 0:
    vit_comparison_results = compare_vit_improvements(X_filled, X_improved_filled, y_class)

    # Display results
    print("\n" + "="*70)
    print("ViT IMPROVEMENT COMPARISON RESULTS")
    print("="*70)
    print(f"{'Feature Set':<30} {'Features':<10} {'CV Accuracy':<15} {'Improvement':<15}")
    print("-"*70)

    baseline_accuracy = None
    for name, result in vit_comparison_results.items():
        if 'Traditional + Linguistic' in name:
            baseline_accuracy = result['cv_mean']

        improvement = ""
        if baseline_accuracy and name != 'Traditional + Linguistic':
            improvement = f"+{(result['cv_mean'] - baseline_accuracy)*100:.1f}%"

        print(f"{name:<30} {result['n_features']:<10} {result['cv_mean']:.3f}±{result['cv_std']:.3f}   {improvement:<15}")

    # Visualize improvement
    if len(vit_comparison_results) > 1:
        plt.figure(figsize=(12, 6))
        names = list(vit_comparison_results.keys())
        accuracies = [vit_comparison_results[name]['cv_mean'] for name in names]
        errors = [vit_comparison_results[name]['cv_std'] for name in names]

        colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold'][:len(names)]
        bars = plt.bar(range(len(names)), accuracies, yerr=errors, capsize=5,
                      color=colors, alpha=0.7, edgecolor='black')

        plt.xlabel('Feature Sets')
        plt.ylabel('Cross-Validation Accuracy')
        plt.title('ViT Feature Improvement Comparison')
        plt.xticks(range(len(names)), names, rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for i, (acc, err) in enumerate(zip(accuracies, errors)):
            plt.text(i, acc + err + 0.01, f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()

        # Feature importance for improved ViT
        if 'Improved ViT' in vit_comparison_results:
            improved_vit_cols = [col for col in X_improved_filled.columns if col.startswith('vit_improved_')]
            if len(improved_vit_cols) > 0:
                X_vit_improved = X_improved_filled[improved_vit_cols]
                rf_vit = RandomForestClassifier(n_estimators=100, random_state=42)
                rf_vit.fit(StandardScaler().fit_transform(X_vit_improved), y_class)

                # Plot top 20 ViT features
                feature_importance = pd.DataFrame({
                    'feature': improved_vit_cols,
                    'importance': rf_vit.feature_importances_
                }).sort_values('importance', ascending=False)

                plt.figure(figsize=(10, 6))
                top_features = feature_importance.head(20)
                plt.barh(range(len(top_features)), top_features['importance'])
                plt.yticks(range(len(top_features)),
                          [f"ViT Component {f.split('_')[-1]}" for f in top_features['feature']])
                plt.xlabel('Feature Importance')
                plt.title('Top 20 Improved ViT Feature Components')
                plt.gca().invert_yaxis()
                plt.tight_layout()
                plt.show()

else:
    print("Improved features not available for comparison.")


### 7.3 ViT vs Traditional Features Performance Comparison


In [ ]:
# Performance comparison between different feature sets
def compare_feature_sets(X_filled, y_class):
    """
    Compare performance of different feature combinations
    """
    feature_sets = {}

    # 1. Traditional audio features only
    traditional_cols = [col for col in X_filled.columns
                       if not col.startswith('vit_')
                       and col not in ['text_length', 'word_count', 'vocabulary_size', 'type_token_ratio',
                                     'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'sentence_count',
                                     'avg_sentence_length', 'flesch_kincaid_grade', 'adj_ratio',
                                     'named_entities', 'disfluencies', 'repetitions']]
    feature_sets['Traditional Audio'] = X_filled[traditional_cols]

    # 2. ViT features only
    vit_cols = [col for col in X_filled.columns if col.startswith('vit_')]
    if len(vit_cols) > 0:
        feature_sets['ViT Features'] = X_filled[vit_cols]

    # 3. Linguistic features only
    ling_cols = [col for col in X_filled.columns
                if col in ['text_length', 'word_count', 'vocabulary_size', 'type_token_ratio',
                          'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'sentence_count',
                          'avg_sentence_length', 'flesch_kincaid_grade', 'adj_ratio',
                          'named_entities', 'disfluencies', 'repetitions']]
    if len(ling_cols) > 0:
        feature_sets['Linguistic Features'] = X_filled[ling_cols]

    # 4. Traditional + Linguistic (without ViT)
    trad_ling_cols = traditional_cols + ling_cols
    feature_sets['Traditional + Linguistic'] = X_filled[trad_ling_cols]

    # 5. All features (including ViT)
    feature_sets['All Features'] = X_filled

    # Test each feature set
    results = {}
    scaler = StandardScaler()

    for name, features in feature_sets.items():
        if features.shape[1] == 0:
            continue

        print(f"\nTesting {name} ({features.shape[1]} features)...")

        # Scale features
        X_scaled = scaler.fit_transform(features)

        # Train Random Forest (best performing model)
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        cv_scores = cross_val_score(rf, X_scaled, y_class, cv=5, scoring='accuracy')

        results[name] = {
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'n_features': features.shape[1]
        }

        print(f"CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    return results

# Run feature comparison
print("Comparing different feature sets...")
comparison_results = compare_feature_sets(X_filled, y_class)

# Display results
print("\n" + "="*60)
print("FEATURE SET COMPARISON RESULTS")
print("="*60)
print(f"{'Feature Set':<25} {'Features':<10} {'CV Accuracy':<15} {'Std Dev':<10}")
print("-"*60)

for name, result in comparison_results.items():
    print(f"{name:<25} {result['n_features']:<10} {result['cv_mean']:.3f}±{result['cv_std']:.3f}")

# Visualize comparison
feature_names = list(comparison_results.keys())
accuracies = [comparison_results[name]['cv_mean'] for name in feature_names]
errors = [comparison_results[name]['cv_std'] for name in feature_names]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(feature_names)), accuracies, yerr=errors, capsize=5, alpha=0.7)
plt.xlabel('Feature Sets')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Performance Comparison: Different Feature Sets')
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (acc, err) in enumerate(zip(accuracies, errors)):
    plt.text(i, acc + err + 0.01, f'{acc:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


### 7.2 Regression Models (MMSE Score Prediction)


In [ ]:
# Define regression models
regression_models = {
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf'),
    'Linear Regression': LinearRegression()
}

# Train and evaluate regression models
regression_results = {}

for name, model in regression_models.items():
    print(f"\nTraining {name}...")

    # Cross-validation
    cv_scores = cross_val_score(model, X_mmse_scaled, y_mmse, cv=5, scoring='r2')

    # Train on training set
    model.fit(X_train_mmse, y_train_mmse)

    # Predictions
    y_pred_mmse = model.predict(X_val_mmse)

    # Evaluation metrics
    mse = mean_squared_error(y_val_mmse, y_pred_mmse)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val_mmse, y_pred_mmse)

    regression_results[name] = {
        'model': model,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'val_mse': mse,
        'val_rmse': rmse,
        'val_r2': r2,
        'predictions': y_pred_mmse
    }

    print(f"CV R²: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")
    print(f"Validation RMSE: {rmse:.3f}")
    print(f"Validation R²: {r2:.3f}")


## 8. Model Evaluation and Visualization


In [ ]:
# Plot confusion matrices for classification
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, results) in enumerate(classification_results.items()):
    cm = confusion_matrix(y_val, results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name}\nAccuracy: {results["val_accuracy"]:.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xticklabels(['Control', 'Dementia'])
    axes[idx].set_yticklabels(['Control', 'Dementia'])

plt.tight_layout()
plt.show()

# Plot regression results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, results) in enumerate(regression_results.items()):
    axes[idx].scatter(y_val_mmse, results['predictions'], alpha=0.6)
    axes[idx].plot([y_val_mmse.min(), y_val_mmse.max()],
                   [y_val_mmse.min(), y_val_mmse.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Actual MMSE')
    axes[idx].set_ylabel('Predicted MMSE')
    axes[idx].set_title(f'{name}\nR² = {results["val_r2"]:.3f}, RMSE = {results["val_rmse"]:.3f}')

plt.tight_layout()
plt.show()


## 9. Test Set Predictions


In [ ]:
# Extract features for test data
print("Extracting features for test data...")
test_features = extract_features_for_dataset(TEST_PATH, test_meta, 'test')

# Prepare test data
X_test = test_features[feature_columns]
X_test_filled = X_test.fillna(X_test.mean())
X_test_scaled = scaler.transform(X_test_filled)

# Select best models
best_class_model = max(classification_results.items(), key=lambda x: x[1]['val_accuracy'])
best_reg_model = max(regression_results.items(), key=lambda x: x[1]['val_r2'])

print(f"Best classification model: {best_class_model[0]}")
print(f"Best regression model: {best_reg_model[0]}")

# Make predictions on test set
test_class_pred = best_class_model[1]['model'].predict(X_test_scaled)
test_mmse_pred = best_reg_model[1]['model'].predict(X_test_scaled)

# Create submission DataFrames
submission_class = pd.DataFrame({
    'ID': test_features['subject_id'],
    'Prediction': test_class_pred
})

submission_mmse = pd.DataFrame({
    'ID': test_features['subject_id'],
    'Prediction': test_mmse_pred
})

# Save submissions in the format required by ADReSS Challenge
submission_class.to_csv('test_results_classification.txt', sep=';', index=False)
submission_mmse.to_csv('test_results_regression.txt', sep=';', index=False)

print("Test predictions completed!")
print(f"Classification predictions: {submission_class['Prediction'].value_counts()}")
print(f"MMSE predictions - Mean: {submission_mmse['Prediction'].mean():.2f}, Std: {submission_mmse['Prediction'].std():.2f}")


In [ ]:
# Fill the original test_results.txt file with predictions
def fill_test_results_file(original_file_path, predictions_dict, output_file_path, task_type='classification'):
    """
    Fill the original test_results.txt file with predictions

    Args:
        original_file_path: Path to the original test_results.txt file
        predictions_dict: Dictionary mapping subject_id to prediction
        output_file_path: Path to save the filled results
        task_type: 'classification' or 'regression'
    """
    try:
        # Read the original file
        with open(original_file_path, 'r') as f:
            lines = f.readlines()

        # Process each line
        filled_lines = []
        for line in lines:
            line = line.strip()
            if line.startswith('ID'):
                # Header line
                filled_lines.append(line + '\n')
            elif line and ';' in line:
                # Data line
                subject_id = line.split(';')[0].strip()
                if subject_id in predictions_dict:
                    prediction = predictions_dict[subject_id]
                    # Format prediction based on task type
                    if task_type == 'classification':
                        pred_str = str(int(prediction))
                    else:  # regression
                        pred_str = f"{prediction:.2f}"
                    filled_lines.append(f"{subject_id} ; {pred_str}\n")
                else:
                    filled_lines.append(line + '\n')
            else:
                filled_lines.append(line + '\n')

        # Write the filled file
        with open(output_file_path, 'w') as f:
            f.writelines(filled_lines)

        print(f"Successfully filled {output_file_path}")

    except Exception as e:
        print(f"Error filling test results file: {e}")

# Create prediction dictionaries
class_pred_dict = dict(zip(test_features['subject_id'], test_class_pred))
mmse_pred_dict = dict(zip(test_features['subject_id'], test_mmse_pred))

# Original test_results.txt file path
original_test_results_path = f'{TEST_PATH}/test_results.txt'

# Fill the test results files with predictions
fill_test_results_file(
    original_test_results_path,
    class_pred_dict,
    'test_results_classification_filled.txt',
    'classification'
)

fill_test_results_file(
    original_test_results_path,
    mmse_pred_dict,
    'test_results_regression_filled.txt',
    'regression'
)

# Display sample results
print("\nSample Classification Results:")
print("Subject ID | Prediction")
print("-" * 25)
for i, (subj_id, pred) in enumerate(list(class_pred_dict.items())[:10]):
    label = "Control" if pred == 0 else "Dementia"
    print(f"{subj_id:10} | {pred} ({label})")

print("\nSample MMSE Regression Results:")
print("Subject ID | MMSE Score")
print("-" * 25)
for i, (subj_id, pred) in enumerate(list(mmse_pred_dict.items())[:10]):
    print(f"{subj_id:10} | {pred:.2f}")

print("\nFiles generated:")
print("- test_results_classification.txt: Standard format classification results")
print("- test_results_regression.txt: Standard format regression results")
print("- test_results_classification_filled.txt: Original format filled with classification predictions")
print("- test_results_regression_filled.txt: Original format filled with regression predictions")


## 10. Results Summary and Clinical Insights


In [ ]:
# Summary of results
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

print("\nCLASSIFICATION RESULTS:")
for name, results in classification_results.items():
    print(f"{name:20} | CV: {results['cv_mean']:.3f} ± {results['cv_std']:.3f} | Val: {results['val_accuracy']:.3f}")

print("\nREGRESSION RESULTS:")
for name, results in regression_results.items():
    print(f"{name:20} | CV R²: {results['cv_mean']:.3f} ± {results['cv_std']:.3f} | Val R²: {results['val_r2']:.3f} | RMSE: {results['val_rmse']:.3f}")

print(f"\nBest Classification Model: {best_class_model[0]}")
print(f"Best Regression Model: {best_reg_model[0]}")

print("\nTEST SET PREDICTIONS:")
print(f"Classification - Control: {(test_class_pred == 0).sum()}, Dementia: {(test_class_pred == 1).sum()}")
print(f"MMSE Regression - Range: [{test_mmse_pred.min():.1f}, {test_mmse_pred.max():.1f}], Mean: {test_mmse_pred.mean():.1f}")

# Visualize test predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Classification predictions distribution
pred_counts = pd.Series(test_class_pred).value_counts()
axes[0].bar(['Control', 'Dementia'], [pred_counts.get(0, 0), pred_counts.get(1, 0)])
axes[0].set_title('Test Set Classification Predictions')
axes[0].set_ylabel('Count')

# MMSE predictions distribution
axes[1].hist(test_mmse_pred, bins=15, alpha=0.7, edgecolor='black')
axes[1].set_title('Test Set MMSE Score Predictions')
axes[1].set_xlabel('Predicted MMSE Score')
axes[1].set_ylabel('Count')
axes[1].axvline(test_mmse_pred.mean(), color='red', linestyle='--', label=f'Mean: {test_mmse_pred.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()


## 11. Conclusion and Future Work

### Key Findings:

1. **Multi-modal Approach**: The combination of audio and linguistic features provides comprehensive information for dementia detection.

2. **Important Feature Categories**:
   - **Audio Features**: MFCC coefficients, spectral features, and prosodic characteristics
   - **Linguistic Features**: Vocabulary diversity, syntactic complexity, and semantic measures
   - **Temporal Features**: Speech timing and pause patterns

3. **Model Performance**: The ensemble methods (Random Forest, Gradient Boosting) showed promising results for both classification and regression tasks.

### Clinical Implications:
- Early detection capability through speech analysis
- Non-invasive assessment method
- Potential for remote monitoring and telemedicine applications
- Objective measurement complementing clinical assessments

### Technical Achievements:
- Comprehensive feature extraction from both audio and text modalities
- Robust cross-validation methodology
- Interpretable machine learning models
- Ready-to-submit predictions for ADReSS Challenge

### Future Improvements:
1. **Deep Learning Models**: Implement CNN/RNN architectures for end-to-end learning
2. **Advanced Audio Features**: Add more sophisticated acoustic features (prosodic patterns, voice quality measures)
3. **Semantic Analysis**: Incorporate word embeddings and semantic similarity measures
4. **Data Augmentation**: Expand dataset with additional speech samples and cross-linguistic validation
5. **Feature Selection**: Apply more advanced feature selection techniques
6. **Ensemble Methods**: Combine multiple models for improved performance

### Limitations:
- Small dataset size limits generalizability
- Limited demographic diversity in the current dataset
- Language-specific features (English only)
- Need for larger validation studies across different populations
- Potential overfitting due to high-dimensional feature space

### Files Generated:
- `train_features.csv`: Extracted training features
- `test_results_classification.txt`: Classification predictions for submission
- `test_results_regression.txt`: MMSE score predictions for submission

This notebook provides a complete pipeline for Alzheimer's dementia detection using the ADReSS Challenge dataset, incorporating state-of-the-art audio and linguistic feature extraction techniques with robust machine learning evaluation.
